In [14]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# QUICK START — run these cells in order
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



install_deps()


✓ Dependencies installed


In [15]:
import ee

ee.Authenticate()
#ee.Initialize(project='geoforest-492210')

True

In [16]:

submit_export_tasks()


✓ GEE initialised  (project: geoforest-492210)
Ghana grid: 117 cells  |  Already submitted: 0
Composite: 2024-06-01 → 2025-12-31



Submitting tasks:   0%|          | 0/117 [00:00<?, ?it/s]

/tmp/ipykernel_55/3593781938.py:214: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "submitted_at": datetime.utcnow().isoformat(),



✓ 117 new tasks submitted  (117 total)
Monitor at: https://code.earthengine.google.com/tasks


In [21]:

check_task_status()  


✓ GEE initialised  (project: geoforest-492210)


Checking tasks:   0%|          | 0/117 [00:00<?, ?it/s]


Task status (117 total):
  OTHER         117  ████████████████████

Progress: 0/117  (0% complete)


## ACTUAL CHECKING OF TAKS STATUS

In [ ]:
import ee, json
from pathlib import Path

authenticate_gee()

tasks_path = Path(TASKS_FILE)
tasks = json.loads(tasks_path.read_text())
counts = {"COMPLETED": 0, "RUNNING": 0, "READY": 0,
          "FAILED": 0, "CANCELLED": 0, "OTHER": 0}

for t in tasks:
    try:
        state = ee.data.getTaskStatus(t["task_id"])[0]["state"]
    except Exception as e:
        state = "OTHER"
    t["status"] = state
    counts[state if state in counts else "OTHER"] += 1

tasks_path.write_text(json.dumps(tasks, indent=2))

total = len(tasks)
for k, v in counts.items():
    if v > 0:
        print(f"  {k:<12} {v:>4} / {total}")

## RUN WHEN ALL TASKS ARE COMPLETED 

In [1]:
# Run this first to install rclone
import subprocess
subprocess.run("curl https://rclone.org/install.sh | sudo bash", shell=True)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4734  100  4734    0     0  22585      0 --:--:-- --:--:-- --:--:-- 22650


Archive:  rclone-current-linux-amd64.zip
   creating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/rclone.1  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/README.html  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/rclone  [binary]
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/git-log.txt  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.73.3-linux-amd64/README.txt  [text]  


Updating index cache for path `/usr/local/man/man1'. Wait...
done.


Purging old database entries in /usr/share/man...
Processing manual pages under /usr/share/man...
Purging old database entries in /usr/share/man/pl...
Processing manual pages under /usr/share/man/pl...
Purging old database entries in /usr/share/man/pt_BR...
Processing manual pages under /usr/share/man/pt_BR...
Purging old database entries in /usr/share/man/tr...
Processing manual pages under /usr/share/man/tr...
Purging old database entries in /usr/share/man/ko...
Processing manual pages under /usr/share/man/ko...
Purging old database entries in /usr/share/man/fr...
Processing manual pages under /usr/share/man/fr...
Purging old database entries in /usr/share/man/da...
Processing manual pages under /usr/share/man/da...
Purging old database entries in /usr/share/man/ja...
Processing manual pages under /usr/share/man/ja...
Purging old database entries in /usr/share/man/es...
Processing manual pages under /usr/share/man/es...
Purging old database entries in /usr/share/man/de...
Processing 

CompletedProcess(args='curl https://rclone.org/install.sh | sudo bash', returncode=0)

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL A — Install & auth (run once)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess
subprocess.run(["pip", "install", "-q", "earthengine-api", "rasterio", "tqdm", "pillow"])

CompletedProcess(args=['pip', 'install', '-q', 'earthengine-api', 'rasterio', 'tqdm', 'pillow'], returncode=0)

In [3]:
import ee
ee.Authenticate()
ee.Initialize(project="geoforest-492210")
print("✓ GEE ready")

✓ GEE ready


In [8]:
import ee, json
from pathlib import Path

# ── Config ────────────────────────────────────────────────────
PROJECT    = "geoforest-492210"
TASKS_FILE = "/kaggle/working/ghana_s2_dataset/gee_tasks.json"
Path(TASKS_FILE).parent.mkdir(parents=True, exist_ok=True)

# ── Init (already authenticated, just re-init to be safe) ────
ee.Initialize(project=PROJECT)

# ── Pull ALL tasks from GEE ───────────────────────────────────
print("Fetching task list from GEE…")
all_tasks = ee.data.listOperations()          # returns up to 500 most recent tasks
print(f"  Total tasks found: {len(all_tasks)}")

# ── Filter to SUCCEEDED (GEE uses 'SUCCEEDED', not 'COMPLETED')
tasks = []
state_counts = {}

for t in all_tasks:
    # GEE Operation format: metadata.state
    meta  = t.get("metadata", {})
    state = meta.get("state", "UNKNOWN")
    state_counts[state] = state_counts.get(state, 0) + 1

    # Accept both naming conventions just in case
    if state in ("SUCCEEDED", "COMPLETED"):
        task_id     = t.get("name", "").split("/")[-1]   # "projects/.../operations/TASK_ID"
        description = meta.get("description", "")
        dest_uris   = meta.get("destinationUris", [])
        tasks.append({
            "task_id":     task_id,
            "status":      "COMPLETED",
            "description": description,
            "dest_uris":   dest_uris,
        })

# ── Save ─────────────────────────────────────────────────────
Path(TASKS_FILE).write_text(json.dumps(tasks, indent=2))

print("\nTask states across all operations:")
for k, v in sorted(state_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<20} {v:>4}")

print(f"\n✓ {len(tasks)} COMPLETED task(s) saved → {TASKS_FILE}")
if tasks:
    print("\nSample entries:")
    for t in tasks[:3]:
        print(f"  {t['task_id']}  |  {t['description'][:60]}")

Fetching task list from GEE…
  Total tasks found: 117

Task states across all operations:
  FAILED                 96
  SUCCEEDED              21

✓ 21 COMPLETED task(s) saved → /kaggle/working/ghana_s2_dataset/gee_tasks.json

Sample entries:
  6KYGNCCXBV7PDX4QXX2453PF  |  ghana_s2_c01r07_2024
  A4YXBTKOFQ2S6B3CW562DL4I  |  ghana_s2_c01r06_2024
  V3XYNMVSXGUUDZPFX5XEET36  |  ghana_s2_c01r05_2024


In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL B — Check status (run repeatedly until all COMPLETED)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json
from pathlib import Path

TASKS_FILE = "/kaggle/working/ghana_s2_dataset/gee_tasks.json"

tasks  = json.loads(Path(TASKS_FILE).read_text())
counts = {"COMPLETED": 0, "RUNNING": 0, "READY": 0,
          "FAILED": 0, "CANCELLED": 0, "OTHER": 0}

for t in tasks:
    try:
        state = ee.data.getTaskStatus(t["task_id"])[0]["state"]
    except Exception:
        state = "OTHER"
    t["status"] = state
    counts[state if state in counts else "OTHER"] += 1

Path(TASKS_FILE).write_text(json.dumps(tasks, indent=2))

total     = len(tasks)
completed = counts["COMPLETED"]
for k, v in counts.items():
    if v > 0:
        bar = "█" * int(20 * v / max(total, 1))
        print(f"  {k:<12} {v:>4}  {bar}")
print(f"\nProgress: {completed}/{total}  ({100*completed//max(total,1)}%)")


  COMPLETED      21  ████████████████████

Progress: 21/21  (100%)


In [4]:
SA_KEY_PATH = "/kaggle/input/datasets/kwabenaobeng/geoforest-key/geoforest-492210-9fef84626194.json"

In [7]:
import json
from pathlib import Path

key = json.loads(Path("/kaggle/input/datasets/kwabenaobeng/geoforest-key/geoforest-492210-9fef84626194.json").read_text())
print("Service account email:", key["client_email"])

Service account email: geoforest@geoforest-492210.iam.gserviceaccount.com


In [8]:
import subprocess

# List folders shared with the service account
result = subprocess.run([
    "rclone", "lsd", "gdrive:",
    "--drive-shared-with-me",
], capture_output=True, text=True)
print("Shared folders:", result.stdout)
print(result.stderr)

Shared folders:            0 2026-04-07 11:21:54        -1 GhanaS2Dataset




## DOWNLOAD TO KAGGLE

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL C — Download GeoTIFFs from Google Drive (service account)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, json, os
from pathlib import Path

GEOTIFF_DIR   = "/kaggle/working/ghana_s2_dataset/geotiffs"
SA_KEY_PATH   = "/kaggle/input/datasets/kwabenaobeng/geoforest-key/geoforest-492210-9fef84626194.json"   # ← upload your service account key here
GDRIVE_FOLDER = "GhanaS2Dataset"                # ← your GEE export folder name in Drive
os.makedirs(GEOTIFF_DIR, exist_ok=True)

# ── Step 1: Install rclone if not present ─────────────────────
result = subprocess.run(["which", "rclone"], capture_output=True, text=True)
if not result.stdout.strip():
    print("Installing rclone…")
    subprocess.run("curl https://rclone.org/install.sh | bash", shell=True, check=True)
else:
    print("✓ rclone already installed")

# ── Step 2: Write rclone config using service account ─────────
# No browser needed — authenticates via the JSON key file
rclone_conf = f"""[gdrive]
type = drive
scope = drive.readonly
service_account_file = {SA_KEY_PATH}
"""
conf_path = Path("/root/.config/rclone/rclone.conf")
conf_path.parent.mkdir(parents=True, exist_ok=True)
conf_path.write_text(rclone_conf)
print("✓ rclone config written (service account)")

# ── Step 3: Verify connection ──────────────────────────────────
print("\nTesting Drive connection…")
test = subprocess.run(
    ["rclone", "lsd", "gdrive:", "--drive-shared-with-me"],
    capture_output=True, text=True
)
if test.returncode != 0:
    print("❌ Connection failed:", test.stderr)
    raise RuntimeError("rclone cannot connect. Check your service account key.")
else:
    print("✓ Connected. Top-level Drive folders:")
    print(test.stdout[:500])

# ── Step 4: Download GeoTIFFs ─────────────────────────────────
print(f"\nDownloading from gdrive:{GDRIVE_FOLDER} → {GEOTIFF_DIR}")
proc = subprocess.run([
    "rclone", "copy",
    f"gdrive:{GDRIVE_FOLDER}",
    GEOTIFF_DIR,
    "--drive-shared-with-me",
    "--progress",
    "--transfers", "8",
    "--drive-chunk-size", "64M",
    "--include", "*.tif",
], text=True)

tifs = list(Path(GEOTIFF_DIR).glob("*.tif"))
print(f"\n✓ {len(tifs)} GeoTIFF(s) downloaded to {GEOTIFF_DIR}")
if len(tifs) == 0:
    print("⚠ No TIFs found — check that GDRIVE_FOLDER matches your actual folder name in Drive")

✓ rclone already installed
✓ rclone config written (service account)

Testing Drive connection…
✓ Connected. Top-level Drive folders:
           0 2026-04-07 11:21:54        -1 GhanaS2Dataset


Transferred:   	          0 B / 7.253 GiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 21
Transferred:            0 / 21, 0%
Elapsed time:         0.5s
Transferring:
 *                      ghana_s2_c00r00_2024.tif:  0% /324.618Mi, 0/s, -
 *                      ghana_s2_c00r02_2024.tif: transferring
 *                      ghana_s2_c00r03_2024.tif:  0% /360.019Mi, 0/s, -
 *                      ghana_s2_c00r01_2024.tif:  0% /259.367Mi, 0/s, -
 *                      ghana_s2_c00r04_2024.tif:  0% /363.005Mi, 0/s, -
 *                      ghana_s2_c00r05_2024.tif:  0% /373.990Mi, 0/s, -
 *                      ghana_s2_c00r06_2024.tif:  0% /380.212Mi, 0/s, -
 *                      ghana_s2_c00r07_2024.tif:  0% /386.468Mi, 0/s, -Transferred:   	   22.750 MiB / 7.253 GiB, 0%, 0 B/s,

In [ ]:
path_to_dataset = '/kaggle/working/ghana_s2_dataset'

## CHECK IF KAGGLE API IS ENABLED 

In [10]:
import os
print(os.path.exists("/root/.kaggle/kaggle.json"))

False


## ADD KAGGLE CREDENTIALS

In [ ]:
import json
from pathlib import Path

Path("/root/.kaggle").mkdir(exist_ok=True)
Path("/root/.kaggle/kaggle.json").write_text(json.dumps({
    "username": "kwabenaobeng",
    "key": "YOUR_KAGGLE_API_KEY"   # ← from kaggle.com → Settings → API
}))
Path("/root/.kaggle/kaggle.json").chmod(0o600)
print("✓ Kaggle credentials set")

## PUBLISH AS A KAGGLE DATASET

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL E — Publish /kaggle/working/ghana_s2_dataset as a
#           permanent Kaggle dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, subprocess
from pathlib import Path

KAGGLE_USERNAME = "kwabenaobeng"          # ← your Kaggle username
DATASET_SLUG    = "ghana-sentinel2-forest"
BASE_DIR        = "/kaggle/working/ghana_s2_dataset"

# ── Write Kaggle dataset metadata ─────────────────────────────
Path(f"{BASE_DIR}/dataset-metadata.json").write_text(json.dumps({
    "title":    "Ghana Sentinel-2 Forest Dataset",
    "id":       f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
    "description": (
        "Sentinel-2 SR (B4/B3/B2/B8) median composite for all of Ghana "
        "(2024-06-01–2025-12-31). 512×512px patches at 10 m/px. "
        "Includes PNG previews and NDVI-based vegetation labels."
    ),
}, indent=2))
print("✓ dataset-metadata.json written")

# ── Check if dataset already exists ───────────────────────────
check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True, text=True
)

if check.returncode != 0 or "No such dataset" in check.stdout:
    cmd    = ["kaggle", "datasets", "create", "-p", BASE_DIR, "--dir-mode", "zip"]
    action = "Creating new dataset"
else:
    cmd    = ["kaggle", "datasets", "version", "-p", BASE_DIR,
              "-m", "Full Ghana GeoTIFF coverage added", "--dir-mode", "zip"]
    action = "Updating existing dataset"

# ── Upload ────────────────────────────────────────────────────
print(f"{action}…  (this may take a few minutes for large files)")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print(f"\n✓ Dataset live at:")
    print(f"  https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

In [ ]:

# process_geotiffs()
# # or if Drive is mounted:
# # process_geotiffs('/content/drive/MyDrive/GhanaS2Dataset')

# upload_to_kaggle()